***

Preparing Workspace

***

In [ ]:
# General
import numpy as np
import pandas as pd
import os
from tqdm import tqdm
import re
from datetime import date
import math
import seaborn as sns

# Plotting
import matplotlib.pyplot as plt
import plotly
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
from plotly.offline import plot
import plotly.subplots as sp
from plotly.subplots import make_subplots
pd.options.display.float_format = '{:.2f}'.format


# Define user
user = os.getlogin()
path_users = os.path.join('C:\\Users', user)

# Set file paths
if user == 'jfontes':
    # Git
    path_git = os.path.join(path_users, 'Documents', 'Projects', 'Regional-Monitoring', 'Indicator_Gen')


path_config0 = os.path.join(path_git, 'config')
path_config  = os.path.join(path_git, 'Data', 'BLS', 'config')


# Set parameters for export file
path_plots = r"\\webmapping-svr\c$\inetpub\wwwroot\monitoring"
print('Export Location: ' + path_plots)

print(user)
print(path_git)


## User defined functions
exec(open(os.path.join(path_config0,     'functions.py')).read())
exec(open(os.path.join(path_config , 'bls_functions.py')).read())
exec(open(os.path.join(path_config0,     'plot.py')).read())



export = False




***

Jobs_1

***

In [ ]:
# Set Indicator
indicator = 'Jobs_1'
plot_name = 'bar'



## Importing ---

file_name1 = f"{indicator} MSA BLS SM.xlsx"
file_name2 = f"{indicator} National BLS CE.xlsx"

df_msa = pd.read_excel(os.path.join(path_plots, 'Data', file_name1), sheet_name = 'MSA')
df_nat = pd.read_excel(os.path.join(path_plots, 'Data', file_name2), sheet_name = 'National')

df_msa = df_msa.rename(columns = {'MSA':'Geography'})
df_nat = df_nat.rename(columns = {'MSA':'Geography'})

df_jobs = pd.concat([df_msa, df_nat])
df_jobs = df_jobs.rename(columns = {'Total Jobs':'Value'})
df_jobs = df_jobs.reset_index(drop = True)


## Oragnizing ---

df_plot = df_jobs.copy()
month = '09'

df_plot = df_plot[df_plot['Sector'] == 'All']
df_plot = df_plot.sort_values(['Geography', 'date_'], ascending = [True, True])
df_plot = df_plot[df_plot['date_'].str.contains(f'{month}-01')]
df_plot = df_plot[~df_plot['date_'].str.contains('2000-01-01')]
df_plot = df_plot[~df_plot['date_'].str.contains(f'20{month}-01-01')]
df_plot['Jobs_GR'] = df_plot['Value'].pct_change()*100
df_plot.loc[df_plot['date_'] == f'2000-{month}-01', 'Jobs_GR'] = np.nan
df_plot.loc[df_plot['Jobs_GR'] == np.inf, 'Jobs_GR'] = np.nan
df_plot = df_plot.sort_values(['Geography', 'date_'], ascending = [True, False])
df_plot = df_plot[~df_plot['Jobs_GR'].isna()]
df_plot = df_plot.reset_index(drop = True)

wm = lambda x: np.average(x, weights = df_plot.loc[x.index, "Value"]) # weighted average

conditions = [   
       df_plot['Geography'].str.contains('Sac|Yuba')
    , ~df_plot['Geography'].str.contains('Sac|Yuba|National')
    ,  df_plot['Geography'].str.contains('National')
             ]
choices = ['SACOG', 'Peer MSA', 'National']
df_plot['Groups'] = np.select(conditions, choices)
df_plot = df_plot.groupby(['date_', 'Groups'], as_index = False).agg(Value = ('Value', 'sum'), Jobs_GR = ('Jobs_GR', wm))
df_plot = df_plot.sort_values(['Groups', 'date_'], ascending = [True, False])

conditions = [   
         df_plot['date_'].str.contains("|".join(str(year) for year in sequence(2000, 2007, 1)))
       , df_plot['date_'].str.contains("|".join(str(year) for year in sequence(2008, 2011, 1)))
       , df_plot['date_'].str.contains("|".join(str(year) for year in sequence(2012, 2019, 1)))
       , df_plot['date_'].str.contains("|".join(str(year) for year in sequence(2020, 2020, 1)))
       , df_plot['date_'].str.contains("|".join(str(year) for year in sequence(2021, 2025, 1)))
             ]
choices = ["Pre Recession<br>(2000-2008)", "Recession<br>(2008-2011)", "Post Recession<br>(2011-2020)", "Covid<br>(2020)", "Post Covid<br>(2020-2025)"]
df_plot["Period"] = np.select(conditions, choices)

df_plot = df_plot.dropna()
df_plot1 = df_plot.groupby(['Groups', 'Period'], as_index = False)['Jobs_GR'].agg(np.mean)
df_plot2 = df_plot.groupby(['Groups'          ], as_index = False)['Jobs_GR'].agg(np.mean)
df_plot2['Period'] = 'Total<br>(2000-2025)'
df_plot = pd.concat([df_plot1, df_plot2])
categories = ["Total<br>(2000-2025)", "Pre Recession<br>(2000-2008)", "Recession<br>(2008-2011)", "Post Recession<br>(2011-2020)", "Covid<br>(2020)", "Post Covid<br>(2020-2025)"]
df_plot['Period_Sort'] = pd.Categorical(df_plot['Period'], categories)
df_plot = df_plot.sort_values(by = ['Groups', 'Period_Sort'], ascending = [True, True])
df_plot = df_plot.drop(['Period_Sort'], axis = 1)
df_plot = df_plot.rename(columns = {'Jobs_GR':'Growth Rate', 'Period':'Time Period'})
df_plot['Growth Rate'] = round(df_plot["Growth Rate"], 2)

display(df_plot.head())


## Plotting ---


fig = px.bar(df_plot, x='Time Period', y='Growth Rate'
             , color='Groups'
             , color_discrete_map=color_map_nat_peermsa
             , barmode='group'
             , hover_name = 'Time Period')
fig.update_yaxes(tick0=0, dtick=2, ticksuffix='%', range = [-7, 4.1])
fig.update_xaxes(tickangle=0)

title = 'Annual Job Growth Comparison: Sacramento, National, and other Mid-Sized Metro Areas (September)'
fig.update_traces(hovertemplate="Growth Rate: %{y}")


plot_agol(fig, export, title, indicator, plot_name, path_plots)


## Needed to install specific version of kaleido package to get export to run
## pip install --upgrade "kaleido==0.1.*"
# path_png = r"\\webmapping-svr\c$\inetpub\wwwroot\monitoring\png"
# fig.write_image(file=os.path.join(path_png, f'{indicator_name}_{plot_name}.png' ),engine='kaleido', scale=1, width=1200, height=600)


Jobs_1
- Includes all sectors
- Subset to September data only
- Calculate percent change in number of jobs from year to year (september to september)
- Calculate weighted average of percent change in number of jobs (annual job growth) across Peer MSA regions and SACOG regions (weighted by number of jobs)
- Calculate simple mean of annual job growth across time periods

***

Jobs_3

***

In [ ]:
# Set Indicator
indicator = 'Jobs_3'
plot_name = 'goods_services'


## Importing ---

file_name1 = f"{indicator} MSA BLS SM.xlsx"
file_name2 = f"{indicator} National BLS CE.xlsx"

df_msa = pd.read_excel(os.path.join(path_plots, 'Data', file_name1), sheet_name = 'Goods and Services')


## Organizing ---
df_plot = df_msa.copy()

df_plot = df_plot.rename(columns = {'MSA':'Geography'})
df_plot['Geography'] = df_plot['Geography'].map(peer_msa_labels)

df_plot = df_plot[df_plot['Sector'] != 'All']
df_plot = df_plot[df_msa['date_'] == '2025-04-01']
df_plot['Year'] = pd.to_datetime(df_plot['date_'])
df_plot['Year'] = df_plot['Year'].dt.year
df_plot = df_plot.drop(['date_'], axis = 1)
df_plot = df_plot.set_index('Year').reset_index()
df_plot = df_plot.sort_values(['Percentage'], ascending=True)
df_plot = df_plot.reset_index(drop=True)
df_plot['Percentage'] = round(df_plot['Percentage']*100, 1)

display(df_plot.head())


## Plotting ---

color_map = {
    'Goods Producing': '#1E90FF'
    , 'Service-Providing':'#9DC209'
}

fig = px.bar(df_plot, y='Geography', x='Percentage'
             , color='Sector'
             , color_discrete_map=color_map
             , orientation='h')

ticktext = []
for geography in df_plot['Geography'].unique():
    if geography in ['Sacramento, CA', 'Yuba City, CA', 'National']:
        ticktext.append(f'<b>{geography}</b>')
    else:
        ticktext.append(geography)

fig.update_layout(yaxis=dict(tickmode='array', tickvals=df_plot['Geography'].unique(), ticktext=ticktext))

title = 'Economic Structure: Share of Goods Production vs Services by Peer Region, 2024'
fig.update_xaxes(tick0=0, dtick=10, ticksuffix='%')
fig.update_traces(hovertemplate='%{x}')
fig.update_layout(legend=dict(orientation="h", yanchor="bottom", y=-0.1,xanchor="right", x=0.55))

plot_agol(fig, export, title, indicator, plot_name, path_plots)


In [ ]:
# Set Indicator
indicator = 'Jobs_3'
plot_name = 'public'


## Importing ---

file_name1 = f"{indicator} MSA BLS SM.xlsx"

df_msa = pd.read_excel(os.path.join(path_plots, 'Data', file_name1), sheet_name = 'Government and Private')



## Organizing ---
df_plot = df_msa.copy()

df_plot = df_plot.rename(columns = {'MSA':'Geography'})
df_plot['Geography'] = df_plot['Geography'].map(peer_msa_labels)

df_plot = df_plot[df_plot['Sector'] != 'All']
df_plot = df_plot[df_msa['date_'] == '2024-04-01']
df_plot['Year'] = pd.to_datetime(df_plot['date_'])
df_plot['Year'] = df_plot['Year'].dt.year
df_plot = df_plot.drop(['date_'], axis = 1)
df_plot = df_plot.set_index('Year').reset_index()
df_plot = df_plot.reset_index(drop = True)


df_plot = df_plot.sort_values('Percentage', ascending=False)
df_plot = df_plot[df_plot['Sector'] == 'Government']
df_plot['Percentage'] = round(df_plot['Percentage']*100, 1)

display(df_plot.head())


## Plotting ---


fig = px.bar(df_plot, y='Geography', x='Percentage'
             , color='Geography'
             , color_discrete_map=color_map_nat_peers
             , orientation='h')

ticktext = []
for geography in df_plot['Geography'].unique():
    if geography in ['Sacramento, CA', 'Yuba City, CA', 'National']:
        ticktext.append(f'<b>{geography}</b>')
    else:
        ticktext.append(geography)
        
fig.update_layout(yaxis=dict(tickmode='array', tickvals=df_plot['Geography'].unique(), ticktext=ticktext))

title = 'Government Share of Total Regional Jobs, 2024'
fig.update_layout(showlegend = False)
fig.update_xaxes(tick0=0, dtick=5, ticksuffix='%')
fig.update_traces(hovertemplate='Government: %{x}')

export=False
plot_agol(fig, export, title, indicator, plot_name, path_plots)


***

Labor_2

***

In [ ]:
# Set Indicator
indicator = 'Labor_2'
plot_name = 'unemployment'


## Importing ---
indicator = 'Labor_2'

file_name = f"{indicator} MSA BLS LA.xlsx"
df_msa = pd.read_excel(os.path.join(path_plots, 'Data', file_name), sheet_name = 'MSA')
df_jobs = pd.read_excel(os.path.join(path_plots, 'Data', 'Jobs_1 MSA BLS SM.xlsx'), sheet_name = 'MSA')


## Organizing ---

df_msa = df_msa.rename(columns = {'MSA_ID':'MSA ID'})

df_jobs = df_jobs[df_jobs['Sector'] == 'All']
df_jobs = df_jobs[['date_', 'MSA ID', 'Total Jobs']].rename(columns = {'Geography':'MSA'})

df_plot = df_msa.copy()
df_plot = df_plot.rename(columns = {'Geography':'MSA'})
df_plot = df_plot.merge(df_jobs, on = ['date_', 'MSA ID'], how = 'left')
df_plot = df_plot[~df_plot['Total Jobs'].isna()]
df_plot['MSA'] = df_plot['MSA'].map(peer_msa_labels)



conditions = [
    (  df_plot['MSA'] ==  'Sacramento, CA'           ) ,
    (  df_plot['MSA'] ==  'Yuba City, CA'            ) ,
    ( ~df_plot['MSA'].str.contains('Yuba|Sacramento'))
]
choices = ['Sacramento, CA', 'Yuba City, CA', 'Peer MSA']
df_plot["Group"] = np.select(conditions, choices)

df_plot['Year'] = pd.to_datetime(df_plot['date_'])
df_plot['Year'] = df_plot['Year'].dt.year

wm = lambda x: np.average(x, weights = df_plot.loc[x.index, "Total Jobs"]) # weighted average
df_plot = df_plot.groupby(['Year', 'Group'], as_index = False).agg(unemployment_rate = ('Unemployment Rate', wm))
df_plot['unemployment_rate'] = round(df_plot['unemployment_rate']*100, 1)
df_plot = df_plot.sort_values(['Year', 'Group'], ascending = [False, True])


df_plot['Sort'] = pd.Categorical(df_plot['Group'], [
    'Sacramento, CA'
         , 'Yuba City, CA'
         , "Peer MSA"
])
    
df_plot = df_plot.sort_values(['Sort', 'Year'], ascending = [True, False])
df_plot = df_plot.drop(['Sort'], axis = 1)

display(df_plot.head())



## Plotting ---

fig = px.line(df_plot, x='Year', y='unemployment_rate', color='Group', markers = True, color_discrete_map=color_map_sac_yuba_peermsa)


title = '<b>Unemployment Rate by MSA</b>'
fig.update_yaxes(dtick=5, ticksuffix='%', range = [0,22])
fig.update_xaxes(dtick=1, range = [1999.5, 2025.5])
fig.update_traces(hovertemplate="%{y}")
fig.update_layout(legend=dict(orientation="h", yanchor="bottom", y=-0.18,xanchor="right", x=0.65))


plot_agol(fig, export, title, indicator, plot_name, path_plots)



Labor_2
- Includes all sectors
- Calculate weighted average of unemployment rate across months and by Peer MSA regions and SACOG regions (weighted by number of jobs)